# 0. Setup (Layered architecture)

## Install Neccersary Packages

The following packages are needed for the program:

* `ipywidgets`: For UI
* `mne`: For data processing
* `numpy`
* `python-picard`: Specific method for ICA during Preprocessing



In [2]:
%pip install ipywidgets
%pip install mne
%pip install numpy matplotlib scipy
%pip install python-picard
%pip install pandas 
%pip install xlrd openpyxl
%pip install pylsl


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## UI

Building the UI utilising `ipywidgets`

In [3]:
from ipywidgets import interact, Dropdown, Text, VBox, HBox, Checkbox, IntSlider

fileUpload = Text(value='data.bdf', description='File Path:')
labelUpload = Text(value='participant_ratings.xls', description='File Path:')
downsampleOnOff = Checkbox(value=True, description='Downsample')
downsampleDropdown = Dropdown(options=['128', '256', '512'], value='128', description='Downsample Rate:')
bandpassOnOff = Checkbox(value=True, description='Bandpass Filter')
icaMethod = Dropdown(options=['fastica', 'infomax', 'picard'], value='fastica', description='ICA Method:')
icaOnOff = Checkbox(value=True, description='ICA')

ui = VBox([
    HBox([fileUpload, labelUpload]),
    HBox([downsampleOnOff, downsampleDropdown]),
    bandpassOnOff,
    HBox([icaOnOff, icaMethod])
])

display(ui)


# 1. Classes

In [5]:
import mne
import pandas as pd
from scipy.signal import decimate
from scipy.fft import rfft, rfftfreq
import numpy as np
from tqdm.auto import tqdm
RANDSEED = 52

### 1.1 Dataloader

In [ ]:
class Dataloader:
  def getData(self, fileUpload):
    raw = mne.io.read_raw_bdf(fileUpload, preload=True)
    rawcopy = raw.copy().pick([ch for ch in raw.ch_names if ch.startswith(("Fp","F","C","P","O","T","Status"))] )
    print(rawcopy.info['ch_names'])
    events = mne.find_events(raw, stim_channel="Status", initial_event=True) #events comes as 3 columnes (sample Index, prev event code, event code), these are used for segmenting
    return rawcopy, events

  def getLabels(self, labelUpload, particpant):
    df = pd.read_excel(labelUpload)
    valencedf = df[['Participant_id','Valence']]
    valence = valencedf[valencedf['Participant_id'] == particpant]    
    return valence


### 1.2 Preprocess

In [7]:
class Preprocessing:
    def __init__(self, icaMethod, randSeed, nComponents = 0.999999 ):
        """
        Initialize the Preprocessing class with ICA method from the UI, random seed, and ncomponents.
        
        Parameters:
        icaMethod (str): The ICA method to use ('fastica', 'infomax', 'picard'). All are options from the UI.
        randSeed (int): The random seed for reproducibility.
        nComponents (float): The smallest number of components required to explain the cumulative variance of the data greater than n_components, see https://mne.tools/stable/generated/mne.preprocessing.ICA.html#mne.preprocessing.ICA
        """
        self.randSeed = randSeed
        self.icaMethod = icaMethod.value
        self.ncomponents = nComponents 
        
    def preprocess(self, raw, downsampleTrue, downsamplerate, bandpassTrue, icaTrue):
        """
        Preprocess the raw EEG data by optionally applying a lowpass filtering (mandatory), downsampling, ICA, and a bandpass filter.
        
        Parameters:
        raw (mne.io.Raw): The raw EEG data to preprocess. This is the return of Dataloader.getData()
        downsampleTrue (bool): Whether to downsample the data.
        downsamplerate (int): The target sampling rate for downsampling.
        bandpassTrue (bool): Whether to apply a bandpass filter.
        icaTrue (bool): Whether to apply ICA for artifact removal.
        
        Returns:
        mne.io.Raw: The preprocessed EEG data.
        """
        rawCopy = raw.copy()
        
        #lowpass:
        rawCopy = rawCopy.filter(l_freq=1.5, h_freq=None)
        
        #Downsample:
        if(downsampleTrue == True):
            rawCopy = rawCopy.resample(sfreq=int(downsamplerate))
        
        #ICA:
        if(icaTrue == True):
            self.ica = mne.preprocessing.ICA(method=self.icaMethod,n_components=self.ncomponents, random_state=self.randSeed, max_iter='auto')
            self.ica.fit(rawCopy)
            eogChannels = [ch for ch in ['Fp1','Fp2'] if ch in rawCopy.ch_names] #using the frontal channels as EOG reference - Fp1 and Fp2 are the only frontal channels in DEAP
            
            if eogChannels:
                eogArtifacts, _ = self.ica.find_bads_eog(rawCopy, ch_name=eogChannels)
            else:
                eogArtifacts = []  
                
            muscleArtifacts, _ = self.ica.find_bads_muscle(rawCopy) 
            
            exclude = (eogArtifacts or []) + (muscleArtifacts or [])
            exclude = [i for i in exclude if i < self.ica.n_components_]   #remove any eog or muscle components
            self.ica.exclude = exclude
            
            rawCopy = self.ica.apply(rawCopy.copy())
        
        #bandpass
        if(bandpassTrue == True):
            rawCopy = rawCopy.filter(l_freq=4, h_freq=45)
            
        return rawCopy
        


### 1.3 Segmentation

In [8]:
class Segmentation:
    def __init__(self, windowSize=4.0, overlap=0.5):
        """
        Initialises class
        parameters:
        windowSize (int) : length of each epoch in seconds. Default is 4 seconds.
        overlap (float): fraction overlap between windows (0.0-0.9) Deafault is 0.5 as it achieved best results in testing.
        """
        self.windowSize = windowSize
        self.overlap = overlap

    def getEpochs(self, raw, events, eventId, labels):
        """
        Segments the raw data into epochs
        
        parameters:
        raw (mne.io.Raw): preprocessed MNE Raw from Preprocessing.preprocess()
        events (np.array): event markers from Dataloader.getData()
        eventId (int): event code to provide marker of dataset eeg collection start (e.g. {4: 'video'} for DEAP data)
        labels (np.array): np.array of labels from Dataloader.getLabels(), same length as number of trials in events
        
        returns:
        X_segments : (n_epochs, n_channels, n_samples) - segmented EEG data
        y_labels : (n_epochs) - labels for each epoch
        """
        sfreq = raw.info['sfreq']
        winSample = int(self.windowSize * sfreq) 
        stepSample = int(winSample * (1 - self.overlap)) 

        tmin, tmax = 0, 60 #trial is from 0 to 60 seconds
        epochs = mne.Epochs(raw, events, event_id={'video': eventId}, tmin=tmin, tmax=tmax,
                            baseline=None, preload=True)
        epochs = epochs.drop_channels(["Status"])
        data = epochs.get_data()   # shape (n_trials, n_channels, n_samples)

        X_segments, y_segments = [], []

        for trial_idx, trial_data in enumerate(data): 
            label = labels[trial_idx]
            n_samples = trial_data.shape[-1]
            start = int(3 * sfreq)  # skip first 3s for baseline
            end = n_samples

            # sliding windows
            for s in range(start, end - winSample + 1, stepSample):
                seg = trial_data[:, s:s+winSample]  # shape (n_channels, winSample)
                X_segments.append(seg)
                y_segments.append(label)

        return np.array(X_segments), np.array(y_segments)

### 1.4 Feature Extraction


In [9]:
class FeatureExtraction:
    def __init__(self):
        """
        Initialize the FeatureExtraction class using the predefined EEG frequency bands national library of medicine: https://www.ncbi.nlm.nih.gov/books/NBK539805/
        """
        self.bands = {
            'delta': (0.5, 4),
            'theta': (4, 7),
            'alpha': (7, 12),
            'beta':  (12, 30),
            'gamma': (30, 45),
        }

    def fftBandPowers(self, seg, samplingRate):
        """
        performs Fast Fourier Transform (FFT) on a single EEG segment and computes the log-transformed band powers
        returns:
        (n_channels * n_bands,) 
        """
        yf = np.abs(rfft(seg, axis=-1)) ** 2
        xf = rfftfreq(seg.shape[-1], 1 / samplingRate)

        features = [np.sum(yf[:, (xf >= low) & (xf <= high)], axis=-1)
                    for low, high in self.bands.values()]
        features = np.array(features)  
        features = np.log(features + 1e-8)

        return np.concatenate(features, axis=-1)  # (n_channels * n_bands,)

    def extract(self, segments, labels, samplingRate):
        """
        Extracts features from all segments using FFT band powers.
        returns:
        segments : (n_epochs, n_channels, n_samples) 
        labels   : (n_epochs,) 
        """
        X = [self.fftBandPowers(seg, samplingRate) for seg in segments]
        X = np.stack(X, axis=0)
        return X, labels

#2. Pipeline

### 2.1 Constructors


In [10]:
dataloader = Dataloader()
preProcessor = Preprocessing(randSeed = RANDSEED, icaMethod = icaMethod)
segmenter = Segmentation(windowSize=4.0, overlap=0.75)
featureExtraction = FeatureExtraction()


### 2.2 Dataflow

In [12]:
allX = []
allYLabels = []
participants  = []

for i in tqdm(range(22, 33)):  # participants range
  print(f"--------------- Processing Participant {i} ---------------")
  fileUpload = f"s{i:02d}.bdf"
  raw, events = dataloader.getData(fileUpload) 
  
  labels = dataloader.getLabels(labelUpload.value, particpant = i) # labels are (particpant id, valence float score from 1 - 9) 40 values for 40 videos with DEAP
  
  praw = preProcessor.preprocess(raw = raw,  
                                    downsampleTrue = downsampleOnOff.value,
                                    downsamplerate = downsampleDropdown.value,
                                    bandpassTrue = bandpassOnOff.value,
                                    icaTrue = icaOnOff.value)
  
  X_segments, y_segments = segmenter.getEpochs(
    raw=praw,
    events=events,
    eventId=4,
    labels=labels['Valence'].to_numpy())
  
  X, ylabels = featureExtraction.extract(
    X_segments,
    y_segments,
    praw.info['sfreq'] )
  
  allX.append(X)
  allYLabels.append(ylabels)
  participants.extend([i] * len(ylabels))


  0%|          | 0/11 [00:00<?, ?it/s]

--------------- Processing Participant 22 ---------------
Extracting EDF parameters from c:\Users\User\Desktop\Uni\Comp4092 Thesis\notebook saves\s22.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1904127  =      0.000 ...  3718.998 secs...
Finding events on: Status
289 events found on stim channel Status
Event IDs: [    1     2     3     4     5     7 65536]
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1.5 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.50
- Lower transition bandwidth: 1.50 Hz (-6 dB cutoff frequency: 0.75 Hz)
- Filter length: 1127 samples (2.201 s)

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detec

C:\Users\User\AppData\Local\Temp\ipykernel_34104\1666700560.py:41: RuntimeWarning: Using n_components=0.999999 (resulting in n_components_=26) may lead to an unstable mixing matrix estimation because the ratio between the largest (32) and smallest (1.1e-05) variances is too large (> 1e6); consider setting n_components=0.999999 or an integer <= 20
  self.ica.fit(rawCopy)


Fitting ICA took 32.1s.
Using EOG channels: Fp1, Fp2
... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 2560 samples (10.000 s)

... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB 

C:\Users\User\AppData\Local\Temp\ipykernel_34104\1666700560.py:49: RuntimeWarning: No sensor positions found. Scores for bad muscle components are only based on the 'slope' criterion.
  muscleArtifacts, _ = self.ica.find_bads_muscle(rawCopy)


...

Fitting ICA took 13.9s.
Using EOG channels: Fp1, Fp2
... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 2560 samples (10.000 s)

... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB 

C:\Users\User\AppData\Local\Temp\ipykernel_34104\1666700560.py:49: RuntimeWarning: No sensor positions found. Scores for bad muscle components are only based on the 'slope' criterion.
  muscleArtifacts, _ = self.ica.find_bads_muscle(rawCopy)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.00 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 423 samples (1.652 s)

Not setting metadata
40 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 40 events and 15361 original time points ...
22 bad epochs dropped
--------------- Processing Participant 24 ---------------
Extracting EDF parameters from c:\Users\User\Desktop\Uni\Comp4092 Thesis\notebook saves\s24.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info struc

C:\Users\User\AppData\Local\Temp\ipykernel_34104\4029340967.py:3: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_bdf(fileUpload, preload=True)
C:\Users\User\AppData\Local\Temp\ipykernel_34104\4029340967.py:3: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_bdf(fileUpload, preload=True)


ValueError: Missing channels from ch_names required by include:
['Status']

In [ ]:
import pickle
with open("RawData22", 'rb') as f:
    rawX = pickle.load(allX)
    rawY = pickle.load(allYLabels)

In [ ]:
import pickle
with open("156ICAFastICA", 'wb') as f:
    pickle.dump(allX,f)
    pickle.dump(allYLabels,f)
    pickle.dump(participants,f)


In [ ]:
import pickle
with open("RawData22", 'wb') as f:
    pickle.dump(allX,f)
    pickle.dump(allYLabels,f)

In [ ]:
np.unique(y_discrete, return_counts=True)

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.utils import shuffle
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# pca = PCA(n_components=0.95)  # keep 95% variance
# X_pca = pca.fit_transform(allX)

y_discrete = (allYLabels >= 5).astype(int)

lda = LDA(n_components=1)
X_lda = lda.fit_transform(allX, y_discrete)

X_lda, y_discrete = shuffle(X_lda, y_discrete, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_lda, y_discrete, test_size=0.2, random_state=42)

scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)

svm_clf = SVC(C=10, kernel='linear', gamma='scale')

svm_clf.fit(X_train, y_train)

y_pred = svm_clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred) # binary classification
f1 = f1_score(y_test, y_pred) # for binary classification
report = classification_report(y_test, y_pred)

print(f"SVM: Accuracy: {acc:.4f}%, Recall :{recall:.4f}%, F1: {f1:.4f}%")
# print(f"Test Accuracy: {acc:.4f}%")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier # or KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler # For data scaling
from sklearn.metrics import accuracy_score # For evaluating classification models

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(allX, y_discrete, test_size=0.3, random_state=42)

# Scale the data (important for distance-based algorithms like KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=2)

knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

kaccuracy = accuracy_score(y_test, y_pred)
krecall = recall_score(y_test, y_pred) # binary classification
kf1 = f1_score(y_test, y_pred) # for binary classification
kreport = classification_report(y_test, y_pred)

print(f"KNN: Accuracy: {kaccuracy:.4f}%, Recall :{krecall:.4f}%, F1: {kf1:.4f}%")
# print(f"Model Accuracy: {accuracy}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
np.unique(y_discrete, return_counts=True)

# Initialize
rf_clf = RandomForestClassifier(
    n_estimators=100,      # number of trees
    max_depth=10,        # let trees grow fully
    random_state=42,
    n_jobs=-1              # use all CPU cores
)

# Train
rf_clf.fit(X_train, y_train)

# Predict
y_pred_rf = rf_clf.predict(X_test)

# Evaluate
rfaccuracy = accuracy_score(y_test, y_pred_rf)
rfrecall = recall_score(y_test, y_pred_rf) # binary classification
rff1 = f1_score(y_test, y_pred_rf) # for binary classification

print(f"RFC: Accuracy: {rfaccuracy:.4f}%, Recall :{rfrecall:.4f}%, F1: {rff1:.4f}%")

# 3. Results

# 4. Testing

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
import numpy as np

# Assuming you already have:
# allX  -> concatenated feature matrix (n_epochs_total × n_features)
# allYLabels -> original continuous labels
# groups -> array of participant IDs, same length as allYLabels

y_discrete = (allYLabels >= 5).astype(int)

logo = LeaveOneGroupOut()

accuracies, recalls, f1s = [], [], []

for train_idx, test_idx in tqdm(logo.split(allX, y_discrete, groups=participants)):
    X_train, X_test = allX[train_idx], allX[test_idx]
    y_train, y_test = y_discrete[train_idx], y_discrete[test_idx]

    # Standardize (fit only on train)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Train model
    clf = SVC(C=10, kernel='rbf', gamma='scale')
    clf.fit(X_train, y_train)

    # Evaluate
    y_pred = clf.predict(X_test)
    accuracies.append(accuracy_score(y_test, y_pred))
    recalls.append(recall_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred))

# Report mean ± std across participants
print(f"LOSO Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"LOSO Recall:   {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
print(f"LOSO F1:       {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

#Raw
#LOSO Accuracy: 0.2785 ± 0.3908
# LOSO Recall:   0.0429 ± 0.1966
# LOSO F1:       0.0441 ± 0.2023

#256ICA
# LOSO Accuracy: 0.2716 ± 0.3826
# LOSO Recall:   0.0440 ± 0.1982
# LOSO F1:       0.0458 ± 0.2030

# 256FICA fixed code
# LOSO Accuracy: 0.3036 ± 0.3462
# LOSO Recall:   0.0800 ± 0.2099
# LOSO F1:       0.1017 ± 0.2424

#RAW Fixed:
# LOSO Accuracy: 0.4785 ± 0.0981
# LOSO Recall:   0.1742 ± 0.1921
# LOSO F1:       0.2515 ± 0.2764

# 256FICA fixed code, small epochs
# LOSO Accuracy: 0.4632 ± 0.0606
# LOSO Recall:   0.6602 ± 0.2347
# LOSO F1:       0.5480 ± 0.1083

# 256FICA fixed code, small epochs, pca, fixed ICA, rbf,  
# LOSO Accuracy: 0.5536 ± 0.1028
# LOSO Recall:   0.9103 ± 0.2496
# LOSO F1:       0.6569 ± 0.1872

# 256FICA fixed code, small epochs, lda, fixed ICA, Linear,  
# 256FICA fixed code, small epochs, lda, fixed ICA, Linear,  
# LOSO Accuracy: 0.5413 ± 0.1006
# LOSO Recall:   0.9639 ± 0.0157
# LOSO F1:       0.6903 ± 0.0898

#updated preprocess pipiline, loging the features
# LOSO Accuracy: 0.5244 ± 0.0759
# LOSO Recall:   0.7367 ± 0.2762
# LOSO F1:       0.5914 ± 0.1749

##updated preprocess pipiline, Not loging the features
# LOSO Accuracy: 0.5452 ± 0.1082
# LOSO Recall:   0.8823 ± 0.2672
# LOSO F1:       0.6397 ± 0.2017

#updated preprocess pipiline, loging the features, 0.75 sliding window
# LOSO Accuracy: 0.5231 ± 0.0781
# LOSO Recall:   0.7151 ± 0.2866
# LOSO F1:       0.5815 ± 0.1767

#128, ICA set to fast
# LOSO Accuracy: 0.5475 ± 0.1305
# LOSO Recall:   0.6189 ± 0.3202
# LOSO F1:       0.5333 ± 0.2318




In [ ]:
# from sklearn.model_selection import LeaveOneGroupOut
# from sklearn.preprocessing import StandardScaler
# from sklearn.svm import SVC
# from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
# import numpy as np

# # Assuming you already have:
# # allX  -> concatenated feature matrix (n_epochs_total × n_features)
# # allYLabels -> original continuous labels
# # groups -> array of participant IDs, same length as allYLabels

# y_discrete = (allYLabels >= 5).astype(int)

# logo = LeaveOneGroupOut()

# accuracies, recalls, f1s = [], [], []

# for train_idx, test_idx in tqdm(logo.split(allX, y_discrete, groups=participants)):
#     X_train, X_test = allX[train_idx], allX[test_idx]
#     y_train, y_test = y_discrete[train_idx], y_discrete[test_idx]

#     # Standardize (fit only on train)
#     scaler = StandardScaler()
#     X_train = scaler.fit_transform(X_train)
#     X_test = scaler.transform(X_test)

#     # Train model
#     clf = SVC(C=10, kernel='rbf', gamma='scale')
#     clf.fit(X_train, y_train)

#     # Evaluate
#     y_pred = clf.predict(X_test)
#     accuracies.append(accuracy_score(y_test, y_pred))
#     recalls.append(recall_score(y_test, y_pred))
#     f1s.append(f1_score(y_test, y_pred))

# # Report mean ± std across participants
# print(f"LOSO Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
# print(f"LOSO Recall:   {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
# print(f"LOSO F1:       {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

# #Raw
# #LOSO Accuracy: 0.2785 ± 0.3908
# # LOSO Recall:   0.0429 ± 0.1966
# # LOSO F1:       0.0441 ± 0.2023

# #256ICA
# # LOSO Accuracy: 0.2716 ± 0.3826
# # LOSO Recall:   0.0440 ± 0.1982
# # LOSO F1:       0.0458 ± 0.2030

# # 256FICA fixed code
# # LOSO Accuracy: 0.3036 ± 0.3462
# # LOSO Recall:   0.0800 ± 0.2099
# # LOSO F1:       0.1017 ± 0.2424

# #RAW Fixed:
# # LOSO Accuracy: 0.4785 ± 0.0981
# # LOSO Recall:   0.1742 ± 0.1921
# # LOSO F1:       0.2515 ± 0.2764